# House Prices: Hyperparameter Tuning

**Part 2** of the HBKU "Data Tools and Applications" Midterm Project.

This notebook demonstrates three hyperparameter tuning strategies on the Kaggle
[House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) dataset:

1. **Grid Search CV** (exhaustive)
2. **Randomized Search CV** (sampling from distributions)
3. **Optuna Bayesian Optimization** (TPE sampler)

All experiments are logged to **Weights & Biases (WandB)**.

## 1. Imports & Configuration

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import uniform, randint
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import optuna
import wandb

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR = os.path.join(BASE_DIR, "..", "data")
PLOT_DIR = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

print("All imports OK")

All imports OK


## 2. Data Loading & Preprocessing

In [2]:
train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train shape: {train_raw.shape}")
print(f"Test  shape: {test_raw.shape}")

# Remove known outliers (large GrLivArea with low price)
train = train_raw[~((train_raw["GrLivArea"] > 4000) & (train_raw["SalePrice"] < 300000))].copy()
print(f"After outlier removal: {train.shape}")

# Log-transform the target
y_train = np.log1p(train["SalePrice"])
train.drop("SalePrice", axis=1, inplace=True)

test_ids = test_raw["Id"]
test = test_raw.drop("Id", axis=1).copy()
train.drop("Id", axis=1, inplace=True)

# Combine for consistent preprocessing
combined = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined shape: {combined.shape}")

Train shape: (1460, 81)
Test  shape: (1459, 80)
After outlier removal: (1458, 81)
Combined shape: (2917, 79)


In [3]:
# --- Fill missing values ---

# Numeric: fill with median
numeric_cols = combined.select_dtypes(include="number").columns
combined[numeric_cols] = combined[numeric_cols].fillna(combined[numeric_cols].median())

# Categorical: fill with "None" (meaning feature absent)
cat_cols = combined.select_dtypes(include="object").columns
combined[cat_cols] = combined[cat_cols].fillna("None")

# --- Ordinal-encode quality columns ---
quality_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
quality_cols = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond",
                "HeatingQC", "KitchenQual", "FireplaceQu",
                "GarageQual", "GarageCond", "PoolQC"]

for col in quality_cols:
    combined[col] = combined[col].map(quality_map).fillna(0).astype(int)

# --- One-hot encode remaining categoricals ---
remaining_cat = [c for c in cat_cols if c not in quality_cols]
combined = pd.get_dummies(combined, columns=remaining_cat, drop_first=True)

# Split back
n_train = len(y_train)
X_train = combined.iloc[:n_train].values
X_test  = combined.iloc[n_train:].values

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Features: {combined.shape[1]}")

X_train: (1458, 235), X_test: (1459, 235)
Features: 235


## 3. WandB Setup

In [4]:
# Read WandB API key from file (not committed to git)
wandb_key_path = os.path.join(BASE_DIR, "..", "wandb.txt")
with open(wandb_key_path) as f:
    wandb_key = f.read().strip()

wandb.login(key=wandb_key)
WANDB_PROJECT = "house-prices-hyperparameter-tuning"
print(f"WandB logged in. Project: {WANDB_PROJECT}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.


wandb: No netrc file found, creating one.


wandb: Appending key for api.wandb.ai to your netrc file: /Users/ali.alsaifi/.netrc


wandb: Currently logged in as: ali-s-a-alsaifi (ali-s-a-alsaifi-hamad-bin-khalifa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB logged in. Project: house-prices-hyperparameter-tuning


## 4. Baseline Models

Establish baseline RMSE using Ridge regression and default XGBRegressor with 5-fold CV.

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def rmse_cv(model, X, y):
    """Return mean RMSE from 5-fold CV (neg_MSE -> RMSE)."""
    scores = cross_val_score(model, X, y, cv=kf,
                             scoring="neg_mean_squared_error")
    return np.sqrt(-scores).mean()

# --- Ridge baseline ---
ridge = Ridge(alpha=10.0)
ridge_rmse = rmse_cv(ridge, X_train, y_train)
print(f"Ridge baseline RMSE: {ridge_rmse:.5f}")

# --- XGBoost default baseline ---
xgb_default = XGBRegressor(n_estimators=500, learning_rate=0.05,
                           random_state=SEED, n_jobs=-1)
xgb_rmse = rmse_cv(xgb_default, X_train, y_train)
print(f"XGBoost default RMSE: {xgb_rmse:.5f}")

# Log baselines to WandB
run = wandb.init(project=WANDB_PROJECT, name="baseline", config={
    "model": "Ridge + XGBoost defaults",
    "ridge_alpha": 10.0,
    "xgb_n_estimators": 500,
    "xgb_learning_rate": 0.05,
})
wandb.log({"ridge_rmse": ridge_rmse, "xgb_default_rmse": xgb_rmse})
wandb.finish()

results = {"Baseline (Ridge)": ridge_rmse, "Baseline (XGBoost)": xgb_rmse}
print("\nBaseline results logged to WandB.")

Ridge baseline RMSE: 0.11537


XGBoost default RMSE: 0.13013


wandb: setting up run y0t8tvpf


wandb: Tracking run with wandb version 0.25.0


wandb: Run data is saved locally in /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part2_hyperparameter_tuning/wandb/run-20260308_035311-y0t8tvpf
wandb: Run `wandb offline` to turn off syncing.


...

wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:       ridge_rmse ▁
wandb: xgb_default_rmse ▁
wandb: 
wandb: Run summary:
wandb:       ridge_rmse 0.11537
wandb: xgb_default_rmse 0.13013
wandb: 


wandb: 🚀 View run baseline at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning/runs/y0t8tvpf
wandb: ⭐️ View project at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260308_035311-y0t8tvpf/logs



Baseline results logged to WandB.


## 5. Grid Search CV

Exhaustive search over a discrete parameter grid on XGBRegressor.

In [6]:
param_grid = {
    "n_estimators": [300, 500, 800],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
}

grid_xgb = XGBRegressor(random_state=SEED, n_jobs=-1)
grid_search = GridSearchCV(
    grid_xgb, param_grid, cv=kf,
    scoring="neg_mean_squared_error",
    verbose=1, n_jobs=-1
)
grid_search.fit(X_train, y_train)

grid_rmse = np.sqrt(-grid_search.best_score_)
grid_best_params = grid_search.best_params_
print(f"\nGrid Search best RMSE: {grid_rmse:.5f}")
print(f"Best params: {grid_best_params}")

# Log to WandB
run = wandb.init(project=WANDB_PROJECT, name="grid-search", config=grid_best_params)
wandb.log({"best_rmse": grid_rmse, "method": "GridSearchCV",
           "total_fits": len(grid_search.cv_results_["mean_test_score"])})
wandb.finish()

results["Grid Search"] = grid_rmse

Fitting 5 folds for each of 54 candidates, totalling 270 fits



Grid Search best RMSE: 0.11899
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 800, 'subsample': 0.8}


wandb: Tracking run with wandb version 0.25.0


wandb: Run data is saved locally in /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part2_hyperparameter_tuning/wandb/run-20260308_035408-zeprgppa
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run grid-search


...

wandb: updating run metadata; uploading summary


wandb: uploading config.yaml


wandb: 
wandb: Run history:
wandb:  best_rmse ▁
wandb: total_fits ▁
wandb: 
wandb: Run summary:
wandb:  best_rmse 0.11899
wandb:     method GridSearchCV
wandb: total_fits 54
wandb: 


wandb: 🚀 View run grid-search at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning/runs/zeprgppa
wandb: ⭐️ View project at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260308_035408-zeprgppa/logs


## 6. Randomized Search CV

Sample 100 random configurations from continuous distributions.

In [7]:
param_distributions = {
    "n_estimators": randint(200, 1200),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.005, 0.195),     # [0.005, 0.2)
    "subsample": uniform(0.6, 0.4),             # [0.6, 1.0)
    "colsample_bytree": uniform(0.5, 0.5),      # [0.5, 1.0)
    "min_child_weight": randint(1, 10),
    "reg_alpha": uniform(0, 1),
    "reg_lambda": uniform(0.5, 1.5),
}

rand_xgb = XGBRegressor(random_state=SEED, n_jobs=-1)
random_search = RandomizedSearchCV(
    rand_xgb, param_distributions,
    n_iter=100, cv=kf,
    scoring="neg_mean_squared_error",
    random_state=SEED, verbose=1, n_jobs=-1
)
random_search.fit(X_train, y_train)

rand_rmse = np.sqrt(-random_search.best_score_)
rand_best_params = random_search.best_params_
print(f"\nRandom Search best RMSE: {rand_rmse:.5f}")
print(f"Best params: {rand_best_params}")

# Log to WandB
run = wandb.init(project=WANDB_PROJECT, name="random-search", config=rand_best_params)
wandb.log({"best_rmse": rand_rmse, "method": "RandomizedSearchCV", "n_iter": 100})
wandb.finish()

results["Random Search"] = rand_rmse

Fitting 5 folds for each of 100 candidates, totalling 500 fits



Random Search best RMSE: 0.11900
Best params: {'colsample_bytree': np.float64(0.6205424161648159), 'learning_rate': np.float64(0.04498383905877235), 'max_depth': 3, 'min_child_weight': 3, 'n_estimators': 884, 'reg_alpha': np.float64(0.60617463445088), 'reg_lambda': np.float64(0.842964208255194), 'subsample': np.float64(0.8686802737623427)}


wandb: setting up run x6cbjehd


wandb: Tracking run with wandb version 0.25.0


wandb: Run data is saved locally in /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part2_hyperparameter_tuning/wandb/run-20260308_035544-x6cbjehd
wandb: Run `wandb offline` to turn off syncing.


...

wandb: updating run metadata; uploading summary


wandb: uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json


wandb: 
wandb: Run history:
wandb: best_rmse ▁
wandb:    n_iter ▁
wandb: 
wandb: Run summary:
wandb: best_rmse 0.119
wandb:    method RandomizedSearchCV
wandb:    n_iter 100
wandb: 


wandb: 🚀 View run random-search at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning/runs/x6cbjehd
wandb: ⭐️ View project at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260308_035544-x6cbjehd/logs


## 7. Optuna Bayesian Optimization (TPE)

Use Optuna's Tree-structured Parzen Estimator for intelligent Bayesian search (50 trials).

In [8]:
run = wandb.init(project=WANDB_PROJECT, name="optuna-bayesian")

def optuna_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 2.0),
        "random_state": SEED,
        "n_jobs": -1,
    }
    model = XGBRegressor(**params)
    scores = cross_val_score(model, X_train, y_train, cv=kf,
                             scoring="neg_mean_squared_error")
    rmse = np.sqrt(-scores).mean()
    wandb.log({"trial": trial.number, "trial_rmse": rmse, **params})
    return rmse

study = optuna.create_study(direction="minimize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_objective, n_trials=50, show_progress_bar=True)

optuna_rmse = study.best_value
optuna_best_params = study.best_params
print(f"\nOptuna best RMSE: {optuna_rmse:.5f}")
print(f"Best params: {optuna_best_params}")

wandb.log({"best_rmse": optuna_rmse, "method": "Optuna-TPE", "n_trials": 50})
wandb.finish()

results["Optuna (Bayesian)"] = optuna_rmse

wandb: setting up run y4zqxxku


wandb: Tracking run with wandb version 0.25.0


wandb: Run data is saved locally in /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part2_hyperparameter_tuning/wandb/run-20260308_035547-y4zqxxku
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run optuna-bayesian


wandb: ⭐️ View project at https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning


...

[I 2026-03-08 04:01:06,012] Trial 49 finished with value: 0.123528211737678 and parameters: {'n_estimators': 1087, 'max_depth': 7, 'learning_rate': 0.006340742689913131, 'subsample': 0.9803863081818511, 'colsample_bytree': 0.605081846741258, 'min_child_weight': 8, 'reg_alpha': 0.0007833071508420824, 'reg_lambda': 1.3671077310048285}. Best is trial 36 with value: 0.11673649966945492.

Optuna best RMSE: 0.11674
Best params: {'n_estimators': 1125, 'max_depth': 4, 'learning_rate': 0.01941454476058786, 'subsample': 0.6815325546565631, 'colsample_bytree': 0.5873279664401843, 'min_child_weight': 6, 'reg_alpha': 1.1426272308283514e-05, 'reg_lambda': 1.47116991159666}


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml


wandb: 
wandb: Run history:
wandb:        best_rmse ▁
wandb: colsample_bytree ▇▅▅▃▃▄▂▆▅▅▆▄▄▆▄▄▅▆▅▄▆█▅▂▄▃▃▃▅▂▁▂▁▁▂▁▂▂▁▂
wandb:    learning_rate ▄▁▂█▁▁▁▄▂▂▃▃▂▄▂▅▂▁▂▃▂▁▃▅▁▂▂▂▂▂▁▁▁▁▁▁▂▁▁▁
wandb:        max_depth █▆▅▇█▂▂▅▇▁▁▁▂▃▂▃▂▁▁▁▃▁▂▁▂▁▁▁▂▆▂▆▃▂▂▃▅▇▅▆
wandb: min_child_weight ▂▃▂▁▁▇▃▁▄▄▅▃▃▆▆▅▆▄▃▄▅▃▄▆▂▃▃▃▂▄▅▅▆▆▆▆▇█▇▆
wandb:     n_estimators ▄▅▃▄▁▅▂▃▁▇██▇▆▆▇▅▇▇▇▆█▇▆▄█▇▇▄▇▅▅▆▆▆▇▃▇▇▇
wandb:           n_jobs ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         n_trials ▁
wandb:     random_state ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        reg_alpha ▁▁▁▁▁▁▁▁▁▄▇█▁▂▁▁▃▃▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:               +4 ...
wandb: 
wandb: Run summary:
wandb:        best_rmse 0.11674
wandb: colsample_bytree 0.60508
wandb:    learning_rate 0.00634
wandb:        max_depth 7
wandb:           method Optuna-TPE
wandb: min_child_weight 8
wandb:     n_estimators 1087
wandb:           n_jobs -1
wandb:         n_trials 50
wandb:     random_state 42
wandb:               +5 ...
wandb: 


wandb: 🚀 View run optuna-bayesian at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning/runs/y4zqxxku
wandb: ⭐️ View project at: https://wandb.ai/ali-s-a-alsaifi-hamad-bin-khalifa-university/house-prices-hyperparameter-tuning
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260308_035547-y4zqxxku/logs


## 8. Comparison of All Methods

In [9]:
# Summary table
comp_df = pd.DataFrame(list(results.items()), columns=["Method", "CV RMSE (log scale)"])
comp_df = comp_df.sort_values("CV RMSE (log scale)").reset_index(drop=True)
print(comp_df.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(comp_df))]
bars = ax.barh(comp_df["Method"], comp_df["CV RMSE (log scale)"],
               color=colors, edgecolor="grey")
ax.set_xlabel("CV RMSE (log-scale SalePrice)", fontsize=11)
ax.set_title("Hyperparameter Tuning: Method Comparison", fontsize=13)
for bar, val in zip(bars, comp_df["CV RMSE (log scale)"]):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f"{val:.5f}", va="center", fontsize=9)
ax.set_xlim(left=comp_df["CV RMSE (log scale)"].min() - 0.005)
plt.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "method_comparison.png"), dpi=150)
plt.show()
print(f"\nBest method: {comp_df.iloc[0]['Method']} with RMSE = {comp_df.iloc[0]['CV RMSE (log scale)']:.5f}")

            Method  CV RMSE (log scale)
  Baseline (Ridge)             0.115371
 Optuna (Bayesian)             0.116736
       Grid Search             0.118989
     Random Search             0.118999
Baseline (XGBoost)             0.130126



Best method: Baseline (Ridge) with RMSE = 0.11537


## 9. Generate Kaggle Submission

Train the best model on the full training set and predict on the test set.

In [10]:
# Pick the best params from whatever method won
best_method = comp_df.iloc[0]["Method"]
if "Grid" in best_method:
    final_params = grid_best_params
elif "Random" in best_method:
    final_params = rand_best_params
elif "Optuna" in best_method:
    final_params = optuna_best_params
else:
    final_params = {"n_estimators": 500, "learning_rate": 0.05}

print(f"Using params from: {best_method}")
print(final_params)

final_model = XGBRegressor(**final_params, random_state=SEED, n_jobs=-1)
final_model.fit(X_train, y_train)

# Predict and reverse log-transform
preds_log = final_model.predict(X_test)
preds = np.expm1(preds_log)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": preds})
submission_path = os.path.join(BASE_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print(f"\nSubmission saved to: {submission_path}")
print(f"Shape: {submission.shape}")
print(f"Id range: {submission['Id'].min()} - {submission['Id'].max()}")
print(f"Price range: ${submission['SalePrice'].min():,.0f} - ${submission['SalePrice'].max():,.0f}")
print(f"All prices > 0: {(submission['SalePrice'] > 0).all()}")
submission.head(10)

Using params from: Baseline (Ridge)
{'n_estimators': 500, 'learning_rate': 0.05}



Submission saved to: /Users/ali.alsaifi/projects/personal-projects/HBKU/Spring-2026/Data Tools and Applications/midterm-project-excercise/part2_hyperparameter_tuning/submission.csv
Shape: (1459, 2)
Id range: 1461 - 2919
Price range: $38,843 - $591,893
All prices > 0: True


,Id,SalePrice
0,1461,126708.804688
1,1462,158270.296875
2,1463,189115.203125
3,1464,192631.406250
4,1465,191734.078125
5,1466,175176.156250
6,1467,174905.890625
7,1468,171118.343750
8,1469,177417.484375
9,1470,123693.804688
